# Exploración del Modelo 2: capacidades heterogéneas con costo binario

En este notebook se documenta la implementación y validación del Modelo 2 del problema de asignación de salas con movilidad entre bloques consecutivos.

El Modelo 2 extiende la versión inicial del problema incorporando dos elementos:

- los cursos pueden tener distintos tamaños;
- las salas pueden tener distintas capacidades.

Se mantiene el supuesto de un solo edificio y una matriz de costos binaria, donde el costo de movilidad es 0 si un estudiante permanece en la misma sala y 1 si cambia de sala.

El propósito de este notebook es mostrar cómo se generaron los datos sintéticos, cómo se validaron, cómo se resolvió el modelo con Gurobi y cómo se compararon los resultados obtenidos con los resultados esperados.


## Objetivo del notebook

Este notebook tiene como objetivo principal verificar que la implementación computacional del Modelo 2 funciona correctamente sobre una batería de tests sintéticos.

En particular, se busca revisar que:

1. Los datos sintéticos estén bien construidos.
2. Se cumpla la conservación de estudiantes entre bloques.
3. Las capacidades de las salas sean consideradas correctamente.
4. El modelo detecte casos infactibles cuando corresponde.
5. La función objetivo entregue el valor esperado.
6. La implementación en Gurobi respete las restricciones del modelo.


## Descripción del Modelo 2

El Modelo 2 corresponde a una versión del problema con capacidades heterogéneas. Esto significa que ya no se asume que todos los cursos tienen el mismo número de estudiantes ni que todas las salas tienen la misma capacidad.

Los principales conjuntos del modelo son:

- $I$: conjunto de cursos del bloque 1.
- $K$: conjunto de cursos del bloque 2.
- $R$: conjunto de salas disponibles.

Los principales parámetros son:

- $n_i$: tamaño del curso $i$ del bloque 1.
- $m_k$: tamaño del curso $k$ del bloque 2.
- $cap_r$: capacidad de la sala $r$.
- $f_{ik}$: cantidad de estudiantes que pasan desde el curso $i$ al curso $k$.
- $f_{iL}$: cantidad de estudiantes del curso $i$ que quedan libres en el segundo bloque.
- $c_{rs}$: costo de moverse desde la sala $r$ hacia la sala $s$.

En esta versión, el costo entre salas es binario:

$$ c_{rs} =
\begin{cases}
0, & \text{si } r = s,\\
1, & \text{si } r \neq s.
\end{cases} 
$$

Por lo tanto, la función objetivo busca minimizar la cantidad total de estudiantes que cambia de sala entre el bloque 1 y el bloque 2.

## Restricciones principales del Modelo 2

El modelo considera las siguientes restricciones principales.

### 1. Asignación única

Cada curso del bloque 1 debe ser asignado exactamente a una sala:


$$\sum_{r \in R} x_{ir} = 1 \quad \forall i \in I$$


Cada curso del bloque 2 también debe ser asignado exactamente a una sala:

$$
\sum_{s \in R} y_{ks} = 1 \quad \forall k \in K
$$

### 2. Ocupación única

Cada sala puede recibir a lo más un curso en cada bloque:


$$\sum_{i \in I} x_{ir} \leq 1 \quad \forall r \in R$$


$$\sum_{k \in K} y_{ks} \leq 1 \quad \forall s \in R$$

Una misma sala sí puede usarse en ambos bloques, porque corresponden a horarios distintos.

### 3. Capacidad suficiente

Esta es la restricción nueva más importante del Modelo 2. Si un curso se asigna a una sala, entonces debe caber en ella:


$$n_i x_{ir} \leq cap_r \quad \forall i \in I, \forall r \in R$$


$$m_k y_{ks} \leq cap_s \quad \forall k \in K, \forall s \in R$$

### 4. Linealización

Para representar correctamente el producto entre asignaciones, se usa la variable auxiliar $z_{ikrs}$, que indica si el curso $i$ está en la sala $r$ y el curso $k$ está en la sala $s$.

La linealización se impone mediante:


$$z_{ikrs} \leq x_{ir}$$


$$z_{ikrs} \leq y_{ks}$$


$$z_{ikrs} \geq x_{ir} + y_{ks} - 1$$

## Estructura de archivos usada

Para mantener el proyecto ordenado, se separó el trabajo en distintos archivos `.py`.

La estructura usada es:

```text
src/
├── generar_data_modelo_2.py
├── validar_data.py
├── modelo_2_gurobi.py
└── correr_tests_modelo_2.py 


Cada archivo tiene una responsabilidad distinta:

* generar_data_modelo_2.py: genera las instancias sintéticas del Modelo 2.
* validar_data.py: revisa que los datos estén bien construidos.
* modelo_2_gurobi.py: implementa y resuelve el modelo usando Gurobi.
* correr_tests_modelo_2.py: ejecuta todos los tests y compara resultados esperados con obtenidos.

Los datos generados se guardan en: 
```text
 data/modelo2/

## Archivos de cada test

Cada carpeta de test contiene los siguientes archivos:

```text
cursos_b1.csv
cursos_b2.csv
salas.csv
flujos.csv
libres.csv
costos.csv
metadata.json

## Descripción de los tests sintéticos

Para validar el Modelo 2 se construyó una batería de tests sintéticos dividida en dos grupos: tests pequeños y tests medianos.

Los **tests pequeños** permiten revisar partes específicas del modelo de forma controlada. Son útiles para detectar errores básicos en la implementación, porque tienen pocas variables y resultados esperados fáciles de justificar.

Los **tests medianos** tienen entre 5 y 8 cursos. Estos permiten mostrar que el modelo también funciona en instancias más representativas, pero que todavía son lo suficientemente simples como para explicar el resultado óptimo esperado.

| Grupo | Test | Propósito | Resultado esperado |
|---|---|---|---|
| Pequeño | M2_T01_factible_costo_0 | Verificar que existe una solución sin movimiento cuando las capacidades lo permiten | OPTIMAL, costo 0 |
| Pequeño | M2_T02_capacidad_cambia_optimo | Verificar que las capacidades pueden forzar una solución más costosa | OPTIMAL, costo 60 |
| Pequeño | M2_T03_infactible_capacidad | Verificar que el modelo detecta ausencia de salas con capacidad suficiente | INFEASIBLE |
| Pequeño | M2_T04_multiples_optimos | Verificar que el modelo acepta múltiples soluciones óptimas | OPTIMAL, costo 30 |
| Pequeño | M2_T05_estudiantes_libres | Verificar que los estudiantes libres no generan costo de movilidad | OPTIMAL, costo 0 |
| Mediano | M2_T06_5_cursos_capacidades_heterogeneas | Validar una instancia con 5 cursos y capacidades heterogéneas | OPTIMAL, costo 20 |
| Mediano | M2_T07_6_cursos_con_libres | Validar una instancia con 6 cursos y estudiantes libres | OPTIMAL, costo 0 |
| Mediano | M2_T08_8_cursos_capacidad_cambia_solucion | Verificar en una instancia de 8 cursos que las capacidades pueden cambiar la solución | OPTIMAL, costo 80 |
| Mediano | M2_T09_8_cursos_multiples_optimos_parciales | Validar una instancia de 8 cursos con múltiples óptimos parciales | OPTIMAL, costo 124 |

Esta estructura permite validar el modelo de forma progresiva. Primero se comprueba que las restricciones básicas funcionen en casos pequeños; luego se revisa que la implementación siga siendo correcta en instancias medianas.

En particular, los tests permiten revisar:

- asignación única de cursos;
- ocupación única de salas;
- restricciones de capacidad;
- conservación de estudiantes;
- tratamiento de estudiantes libres;
- detección de infactibilidad;
- manejo de múltiples soluciones óptimas;
- cálculo correcto de la función objetivo.

## Validación de datos

Antes de resolver el modelo con Gurobi, se valida que los datos sintéticos estén bien construidos.

La validación revisa principalmente tres aspectos.

### 1. Formato básico

Se revisa que los archivos tengan las columnas necesarias. Por ejemplo:

- `curso_id` y `tamano` en los archivos de cursos.
- `sala_id` y `capacidad` en el archivo de salas.
- `curso_b1`, `curso_b2` y `flujo` en el archivo de flujos.
- `curso_b1` y `libres` en el archivo de estudiantes libres.

### 2. Valores no negativos

Se revisa que no existan tamaños, capacidades, flujos o estudiantes libres con valores negativos.

### 3. Conservación de estudiantes

Para cada curso del bloque 1 debe cumplirse:


$$\sum_{k \in K} f_{ik} + f_{iL} = n_i$$

Esto significa que todos los estudiantes del curso \(i\) del bloque 1 deben ir a algún curso del bloque 2 o quedar libres.

Para cada curso del bloque 2 debe cumplirse:


$$\sum_{i \in I} f_{ik} = m_k$$

Esto significa que el tamaño de cada curso del bloque 2 debe coincidir con la cantidad de estudiantes que llegan a él desde cursos del bloque 1.

También se revisa la consistencia global:


$$\sum_i n_i = \sum_k m_k + \sum_i f_{iL}$$

Si estas condiciones no se cumplen, la instancia está mal construida y no debería resolverse con Gurobi.

## Matriz de costos

En el Modelo 2 se utiliza una matriz de costos binaria. Esto significa que solo se distingue entre quedarse en la misma sala o cambiar de sala.

La regla usada es:

$$
c_{rs} =
\begin{cases}
0, & \text{si } r = s,\\
1, & \text{si } r \neq s.
\end{cases}
$$

Por ejemplo, si se tienen tres salas $r1$, $r2$ y $r3$, la matriz de costos es:

| | r1 | r2 | r3 |
|---|---:|---:|---:|
| r1 | 0 | 1 | 1 |
| r2 | 1 | 0 | 1 |
| r3 | 1 | 1 | 0 |

Esta matriz se genera automáticamente en `generar_data_modelo_2.py` y se guarda como `costos.csv` dentro de cada carpeta de test.

En modelos posteriores, esta matriz podrá cambiar. Por ejemplo, en el Modelo 3 el costo dependerá de si el estudiante se mantiene en el mismo edificio o cambia de edificio.

## Implementación en Gurobi

La implementación del Modelo 2 se encuentra en el archivo:

```text
src/modelo_2_gurobi.py


La función principal es: resolver_modelo_2

Esta función realiza los siguientes pasos:

1) Lee y valida los datos de la carpeta del test.
2) Lee la matriz de costos desde costos.csv.
3) Crea el modelo en Gurobi.
4) Define las variables de decisión x, y y z.
5) Agrega la función objetivo.
6) Agrega las restricciones de asignación única.
7) Agrega las restricciones de ocupación única.
8) Agrega las restricciones de capacidad.
9) Agrega las restricciones de linealización.
10) Optimiza el modelo.
11) Devuelve el status, el valor objetivo y las asignaciones obtenidas.


## 1. Preparación del notebook

Primero importamos las librerías necesarias y conectamos el notebook con los archivos `.py` ubicados en la carpeta `src`.

Este notebook usará funciones ya implementadas en:

- `generar_data_modelo_2.py`
- `validar_data.py`
- `modelo_2_gurobi.py`

In [4]:
from pathlib import Path
import sys
import json  
import pandas as pd  

# Detectar la carpeta principal del proyecto
ROOT = Path.cwd()

# Si estamos dentro de la carpeta notebooks, subimos un nivel
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
DATA_MODELO_2 = ROOT / "data" / "modelo2"

# Agregar src al path para poder importar nuestros archivos .py
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("Carpeta principal:", ROOT)
print("Carpeta src:", SRC)
print("Carpeta data/modelo2:", DATA_MODELO_2)

Carpeta principal: c:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II
Carpeta src: c:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\src
Carpeta data/modelo2: c:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\data\modelo2


In [5]:
from validar_data import validar_instancia_modelo_2
from modelo_2_gurobi import resolver_modelo_2
from generar_data_modelo_2 import generar_tests_pequenos
from generar_data_modelo_2 import generar_tests_medianos
from generar_data_modelo_2 import generar_todos_los_tests


print("Funciones importadas correctamente.")

Funciones importadas correctamente.


## 2. Generación de datos sintéticos

Los datos sintéticos del Modelo 2 se generan con el archivo:

```text
src/generar_data_modelo_2.py

In [6]:
generar_tests_pequenos()

Test generado: M2_T01_factible_costo_0
Test generado: M2_T02_capacidad_cambia_optimo
Test generado: M2_T03_infactible_capacidad
Test generado: M2_T04_multiples_optimos
Test generado: M2_T05_estudiantes_libres

Tests pequeños del Modelo 2 generados correctamente.


## 3. Tests disponibles

A continuación se revisan las carpetas de tests que existen dentro de `data/modelo_2`.

In [7]:
carpetas_tests_pequenos = sorted([
    carpeta for carpeta in DATA_MODELO_2.iterdir()
    if carpeta.is_dir() and carpeta.name.startswith(("M2_T01", "M2_T02", "M2_T03", "M2_T04", "M2_T05"))
])

for carpeta in carpetas_tests_pequenos:
    print(carpeta.name)


M2_T01_factible_costo_0
M2_T02_capacidad_cambia_optimo
M2_T03_infactible_capacidad
M2_T04_multiples_optimos
M2_T05_estudiantes_libres


## 4. Tabla resumen de los tests

Cada test tiene un archivo `metadata.json` con su descripción, status esperado y objetivo esperado.


In [8]:
resumen_tests = []

for carpeta in carpetas_tests_pequenos:
    ruta_metadata = carpeta / "metadata.json"
    
    with open(ruta_metadata, "r", encoding="utf-8") as archivo:
        metadata = json.load(archivo)
    
    resumen_tests.append({
        "test": carpeta.name,
        "descripcion": metadata["descripcion"],
        "status_esperado": metadata["expected_status"],
        "objetivo_esperado": metadata["expected_obj"]
    })

tabla_tests = pd.DataFrame(resumen_tests)
tabla_tests

,test,descripcion,status_esperado,objetivo_esperado
0,M2_T01_factible_costo_0,Caso factible donde todos los estudiantes pued...,OPTIMAL,0.0
1,M2_T02_capacidad_cambia_optimo,La capacidad obliga a usar una asignación más ...,OPTIMAL,60.0
2,M2_T03_infactible_capacidad,"Un curso tiene tamaño 60, pero la sala más gra...",INFEASIBLE,NaN
3,M2_T04_multiples_optimos,Existen varias asignaciones óptimas con el mis...,OPTIMAL,30.0
4,M2_T05_estudiantes_libres,Parte de los estudiantes queda libre y no gene...,OPTIMAL,0.0


## 5. Inspección de un test pequeño específico

Ahora se revisa con más detalle una instancia particular.

En este caso se analizará el test `M2_T02_capacidad_cambia_optimo`, porque permite observar el efecto de las restricciones de capacidad.

In [9]:
nombre_test = "M2_T02_capacidad_cambia_optimo"
carpeta_test = DATA_MODELO_2 / nombre_test

cursos_b1 = pd.read_csv(carpeta_test / "cursos_b1.csv")
cursos_b2 = pd.read_csv(carpeta_test / "cursos_b2.csv")
salas = pd.read_csv(carpeta_test / "salas.csv")
flujos = pd.read_csv(carpeta_test / "flujos.csv")
libres = pd.read_csv(carpeta_test / "libres.csv")
costos = pd.read_csv(carpeta_test / "costos.csv")

print("Test seleccionado:", nombre_test)

Test seleccionado: M2_T02_capacidad_cambia_optimo


### Cursos del bloque 1


In [10]:
cursos_b1

,curso_id,tamano
0,i1,50
1,i2,30


### Cursos del bloque 2

In [11]:
cursos_b2

,curso_id,tamano
0,k1,30
1,k2,50


### Salas disponibles

In [12]:
salas

,sala_id,capacidad
0,r1,50
1,r2,30


### Matriz de flujos

El archivo `flujos.csv` está en formato largo. Para entenderlo mejor, se transforma a formato matriz.

Cada fila representa un curso del bloque 1 y cada columna representa un curso del bloque 2.

In [13]:
F = flujos.pivot(
    index="curso_b1",
    columns="curso_b2",
    values="flujo"
).fillna(0)

F

curso_b2,k1,k2
curso_b1,,
i1,30.0,20.0
i2,0.0,30.0


## 6. Validación de datos

Antes de resolver el modelo con Gurobi, se valida que la instancia esté bien construida.

In [14]:
datos_validados = validar_instancia_modelo_2(carpeta_test)

Validación completada correctamente.

Resumen:
  Cursos bloque 1: 2
  Cursos bloque 2: 2
  Salas: 2
  Total estudiantes bloque 1: 80
  Total estudiantes bloque 2: 80
  Total estudiantes libres: 0
  Flujo total entre cursos: 80


## 7. Resolución del test con Gurobi

Ahora se resuelve el test seleccionado usando la función `resolver_modelo_2`.

Se usa `mostrar_output=False` para ocultar el log largo de Gurobi.

In [15]:
resultado = resolver_modelo_2(carpeta_test, mostrar_output=False)
resultado

Validación completada correctamente.

Resumen:
  Cursos bloque 1: 2
  Cursos bloque 2: 2
  Salas: 2
  Total estudiantes bloque 1: 80
  Total estudiantes bloque 2: 80
  Total estudiantes libres: 0
  Flujo total entre cursos: 80
Set parameter Username
Set parameter LicenseID to value 2808412
Academic license - for non-commercial use only - expires 2027-04-16


{'status': 'OPTIMAL',
 'objetivo': 60.0,
 'asignacion_b1': {'i1': 'r1', 'i2': 'r2'},
 'asignacion_b2': {'k1': 'r2', 'k2': 'r1'}}

### Asignación obtenida para el bloque 1

La siguiente tabla muestra en qué sala queda asignado cada curso del bloque 1.

In [16]:
pd.DataFrame(
    list(resultado["asignacion_b1"].items()),
    columns=["curso_b1", "sala_asignada"]
)

,curso_b1,sala_asignada
0,i1,r1
1,i2,r2


### Asignación obtenida para el bloque 2

La siguiente tabla muestra en qué sala queda asignado cada curso del bloque 2.

In [17]:
pd.DataFrame(
    list(resultado["asignacion_b2"].items()),
    columns=["curso_b2", "sala_asignada"]
)

,curso_b2,sala_asignada
0,k1,r2
1,k2,r1


## 8. Interpretación del test seleccionado

En el test `M2_T02_capacidad_cambia_optimo`, la restricción de capacidad obliga a que los cursos grandes sean asignados a la sala de mayor capacidad.

Por esta razón, el modelo no puede elegir simplemente la asignación que minimiza movilidad ignorando capacidades.

El resultado esperado para este test es:

```text
Status esperado: OPTIMAL
Objetivo esperado: 60

## 9. Tests medianos del Modelo 2

Después de validar los tests pequeños, se agregan tests medianos con entre 5 y 8 cursos.

Estos tests buscan mostrar que el modelo no solo funciona en instancias mínimas, sino también en casos mas representativos.
Para cada instancia se muestra una breve explicación del resultado esperado y luego se muestra el resultado obtenido. Demostrando que el modelo esta bien implementado.


In [18]:
generar_tests_medianos()

Test generado: M2_T06_5_cursos_capacidades_heterogeneas
Test generado: M2_T07_6_cursos_con_libres
Test generado: M2_T08_8_cursos_capacidad_cambia_solucion
Test generado: M2_T09_8_cursos_multiples_optimos_parciales

Tests medianos del Modelo 2 generados correctamente.


In [19]:
carpetas_tests_medianos = sorted([
    carpeta for carpeta in DATA_MODELO_2.iterdir()
    if carpeta.is_dir() and carpeta.name.startswith(("M2_T06", "M2_T07", "M2_T08", "M2_T09"))
])

In [20]:
resumen_tests_medianos = []

for carpeta in carpetas_tests_medianos:
    ruta_metadata = carpeta / "metadata.json"
    
    with open(ruta_metadata, "r", encoding="utf-8") as archivo:
        metadata = json.load(archivo)
    
    resumen_tests_medianos.append({
        "test": carpeta.name,
        "descripcion": metadata["descripcion"],
        "status_esperado": metadata["expected_status"],
        "objetivo_esperado": metadata["expected_obj"]
    })

tabla_tests_medianos = pd.DataFrame(resumen_tests_medianos)
tabla_tests_medianos

,test,descripcion,status_esperado,objetivo_esperado
0,M2_T06_5_cursos_capacidades_heterogeneas,Test mediano con 5 cursos y capacidades hetero...,OPTIMAL,20
1,M2_T07_6_cursos_con_libres,Test mediano con 6 cursos donde todos los curs...,OPTIMAL,0
2,M2_T08_8_cursos_capacidad_cambia_solucion,Test mediano con 8 cursos donde las capacidade...,OPTIMAL,80
3,M2_T09_8_cursos_multiples_optimos_parciales,Test mediano con 8 cursos donde existen múltip...,OPTIMAL,124


In [21]:
from IPython.display import display

def mostrar_resultado_y_asignacion(nombre_test):
    """
    Resuelve un test con Gurobi y muestra:
    - status
    - objetivo
    - asignación bloque 1
    - asignación bloque 2
    """

    carpeta_test = DATA_MODELO_2 / nombre_test

    resultado = resolver_modelo_2(carpeta_test, mostrar_output=False)

    print("Test:", nombre_test)
    print("Status obtenido:", resultado["status"])
    print("Objetivo obtenido:", resultado["objetivo"])

    asignacion_b1 = pd.DataFrame(
        list(resultado["asignacion_b1"].items()),
        columns=["curso_b1", "sala_asignada"]
    )

    asignacion_b2 = pd.DataFrame(
        list(resultado["asignacion_b2"].items()),
        columns=["curso_b2", "sala_asignada"]
    )

    print("\nAsignación bloque 1:")
    display(asignacion_b1)

    print("Asignación bloque 2:")
    display(asignacion_b2)

    return resultado

### M2_T06: 5 cursos con capacidades heterogéneas

Este test valida una instancia mediana con 5 cursos, 5 salas y capacidades distintas. El objetivo es verificar que el modelo respeta las capacidades y encuentra la asignación esperada.

En el bloque 1, las capacidades fuerzan la asignación:

```text
i1 → r1
i2 → r2
i3 → r3
i4 → r4
i5 → r5

En el bloque 2, el curso k2 tiene tamaño 50, por lo que debe ir a r1. Luego k1 queda en r2, y el resto queda en salas del mismo tamaño:

```text
k2 → r1
k1 → r2
k3 → r3
k4 → r4
k5 → r5

El flujo total es 180 y los estudiantes que permanecen en la misma sala son 160. Por lo tanto, el costo esperado es:

$$ 180−160=20$$

In [22]:

resultado_t06 = mostrar_resultado_y_asignacion(
    "M2_T06_5_cursos_capacidades_heterogeneas"
)

Validación completada correctamente.

Resumen:
  Cursos bloque 1: 5
  Cursos bloque 2: 5
  Salas: 5
  Total estudiantes bloque 1: 180
  Total estudiantes bloque 2: 180
  Total estudiantes libres: 0
  Flujo total entre cursos: 180
Test: M2_T06_5_cursos_capacidades_heterogeneas
Status obtenido: OPTIMAL
Objetivo obtenido: 20.0

Asignación bloque 1:


,curso_b1,sala_asignada
0,i1,r1
1,i2,r2
2,i3,r3
3,i4,r4
4,i5,r5


Asignación bloque 2:


,curso_b2,sala_asignada
0,k1,r2
1,k2,r1
2,k3,r3
3,k4,r4
4,k5,r5


### M2_T07: 6 cursos con estudiantes libres

Este test valida una instancia mediana con 6 cursos donde todos los cursos del bloque 1 dejan estudiantes libres.

Los estudiantes libres se usan para conservar la cantidad total de estudiantes, pero no generan costo de movilidad porque no asisten a un curso en el bloque 2.

En esta instancia, los flujos principales son:

```text
i1 → k1
i2 → k2
i3 → k3
i4 → k4
i5 → k5
i6 → k6

La asignación esperada permite que todos los estudiantes que sí continúan a un curso del bloque 2 permanezcan en la misma sala:

```text
i1 y k1 → r1
i2 y k2 → r2
i3 y k3 → r3
i4 y k4 → r4
i5 y k5 → r5
i6 y k6 → r6

Como todos los estudiantes que continúan se quedan en la misma sala, el costo esperado es 0.

In [23]:
resultado_t07 = mostrar_resultado_y_asignacion(
    "M2_T07_6_cursos_con_libres"
)

Validación completada correctamente.

Resumen:
  Cursos bloque 1: 6
  Cursos bloque 2: 6
  Salas: 6
  Total estudiantes bloque 1: 225
  Total estudiantes bloque 2: 195
  Total estudiantes libres: 30
  Flujo total entre cursos: 195
Test: M2_T07_6_cursos_con_libres
Status obtenido: OPTIMAL
Objetivo obtenido: 0.0

Asignación bloque 1:


,curso_b1,sala_asignada
0,i1,r1
1,i2,r2
2,i3,r3
3,i4,r4
4,i5,r5
5,i6,r6


Asignación bloque 2:


,curso_b2,sala_asignada
0,k1,r1
1,k2,r2
2,k3,r3
3,k4,r4
4,k5,r5
5,k6,r6


### M2_T08: 8 cursos donde la capacidad cambia la solución

Este test valida una instancia mediana más grande, con 8 cursos y 8 salas. Es útil para mostrar que el modelo sigue funcionando en un caso más representativo.

Las capacidades fuerzan casi completamente la asignación. En el bloque 1, los cursos quedan ordenados por tamaño:

```text
i1 → r1
i2 → r2
i3 → r3
i4 → r4
i5 → r5
i6 → r6
i7 → r7
i8 → r8

En el bloque 2, k2 tiene tamaño 60 y debe ir a r1, mientras que k1 tiene tamaño 55 y queda en r2:

```text
k2 → r1
k1 → r2
k3 → r3
k4 → r4
k5 → r5
k6 → r6
k7 → r7
k8 → r8

El flujo total es 310 y los estudiantes que permanecen en la misma sala suman 230. Por lo tanto, el costo esperado es:

$$ 310−230=80 $$

In [24]:
resultado_t08 = mostrar_resultado_y_asignacion(
    "M2_T08_8_cursos_capacidad_cambia_solucion"
)

Validación completada correctamente.

Resumen:
  Cursos bloque 1: 8
  Cursos bloque 2: 8
  Salas: 8
  Total estudiantes bloque 1: 310
  Total estudiantes bloque 2: 310
  Total estudiantes libres: 0
  Flujo total entre cursos: 310
Test: M2_T08_8_cursos_capacidad_cambia_solucion
Status obtenido: OPTIMAL
Objetivo obtenido: 80.0

Asignación bloque 1:


,curso_b1,sala_asignada
0,i1,r1
1,i2,r2
2,i3,r3
3,i4,r4
4,i5,r5
5,i6,r6
6,i7,r7
7,i8,r8


Asignación bloque 2:


,curso_b2,sala_asignada
0,k1,r2
1,k2,r1
2,k3,r3
3,k4,r4
4,k5,r5
5,k6,r6
6,k7,r7
7,k8,r8


### M2_T09: 8 cursos con múltiples óptimos parciales

Este test valida una instancia mediana con múltiples soluciones óptimas.

Los cursos están organizados en grupos de dos. Dentro de cada grupo, los cursos tienen el mismo tamaño y los flujos están empatados. Por eso, no se exige una asignación exacta única.

Por ejemplo, en el primer grupo:

```text
i1 e i2 tienen tamaño 40
k1 y k2 tienen tamaño 40
r1 y r2 tienen capacidad 40

Los flujos dentro del grupo son simétricos, por lo que Gurobi puede asignar los cursos de varias formas distintas sin cambiar el valor objetivo.

En este test se espera que los cursos queden dentro de su grupo de salas correspondiente:

```text
i1, i2, k1, k2 → r1 o r2
i3, i4, k3, k4 → r3 o r4
i5, i6, k5, k6 → r5 o r6
i7, i8, k7, k8 → r7 o r8

El flujo total es 248 y los estudiantes que pueden permanecer en la misma sala suman 124. Por lo tanto, el costo esperado es:

$$ 248−124=124$$

In [25]:
resultado_t09 = mostrar_resultado_y_asignacion(
    "M2_T09_8_cursos_multiples_optimos_parciales"
)

Validación completada correctamente.

Resumen:
  Cursos bloque 1: 8
  Cursos bloque 2: 8
  Salas: 8
  Total estudiantes bloque 1: 248
  Total estudiantes bloque 2: 248
  Total estudiantes libres: 0
  Flujo total entre cursos: 248
Test: M2_T09_8_cursos_multiples_optimos_parciales
Status obtenido: OPTIMAL
Objetivo obtenido: 124.0

Asignación bloque 1:


,curso_b1,sala_asignada
0,i1,r1
1,i2,r2
2,i3,r4
3,i4,r3
4,i5,r5
5,i6,r6
6,i7,r7
7,i8,r8


Asignación bloque 2:


,curso_b2,sala_asignada
0,k1,r1
1,k2,r2
2,k3,r4
3,k4,r3
4,k5,r6
5,k6,r5
6,k7,r8
7,k8,r7


In [26]:
def validar_grupos_t09(resultado):
    grupos_b1 = {
        "i1": {"r1", "r2"},
        "i2": {"r1", "r2"},
        "i3": {"r3", "r4"},
        "i4": {"r3", "r4"},
        "i5": {"r5", "r6"},
        "i6": {"r5", "r6"},
        "i7": {"r7", "r8"},
        "i8": {"r7", "r8"},
    }

    grupos_b2 = {
        "k1": {"r1", "r2"},
        "k2": {"r1", "r2"},
        "k3": {"r3", "r4"},
        "k4": {"r3", "r4"},
        "k5": {"r5", "r6"},
        "k6": {"r5", "r6"},
        "k7": {"r7", "r8"},
        "k8": {"r7", "r8"},
    }

    for curso, salas_validas in grupos_b1.items():
        if resultado["asignacion_b1"][curso] not in salas_validas:
            return False

    for curso, salas_validas in grupos_b2.items():
        if resultado["asignacion_b2"][curso] not in salas_validas:
            return False

    return True


print("Asignación válida por grupos:", validar_grupos_t09(resultado_t09))

Asignación válida por grupos: True


## 10. Ejecución de todos los tests

Ahora se resuelven todas las carpetas de tests y se comparan los resultados obtenidos contra los resultados esperados definidos en `metadata.json`.

In [27]:
resultados_tests = []
carpetas_tests = carpetas_tests_pequenos + carpetas_tests_medianos

for carpeta in carpetas_tests:
    with open(carpeta / "metadata.json", "r", encoding="utf-8") as archivo:
        metadata = json.load(archivo)
    
    resultado = resolver_modelo_2(carpeta, mostrar_output=False)
    
    status_esperado = metadata["expected_status"]
    status_obtenido = resultado["status"]
    
    objetivo_esperado = metadata["expected_obj"]
    objetivo_obtenido = resultado["objetivo"]
    
    status_ok = status_esperado == status_obtenido
    
    if status_esperado == "INFEASIBLE":
        objetivo_ok = True
    else:
        objetivo_ok = abs(objetivo_obtenido - objetivo_esperado) <= 1e-6
    
    pasa_test = status_ok and objetivo_ok
    
    resultados_tests.append({
        "test": carpeta.name,
        "status_esperado": status_esperado,
        "status_obtenido": status_obtenido,
        "objetivo_esperado": objetivo_esperado,
        "objetivo_obtenido": objetivo_obtenido,
        "pasa_test": pasa_test
    })

tabla_resultados = pd.DataFrame(resultados_tests)
tabla_resultados

Validación completada correctamente.

Resumen:
  Cursos bloque 1: 3
  Cursos bloque 2: 3
  Salas: 3
  Total estudiantes bloque 1: 90
  Total estudiantes bloque 2: 90
  Total estudiantes libres: 0
  Flujo total entre cursos: 90
Validación completada correctamente.

Resumen:
  Cursos bloque 1: 2
  Cursos bloque 2: 2
  Salas: 2
  Total estudiantes bloque 1: 80
  Total estudiantes bloque 2: 80
  Total estudiantes libres: 0
  Flujo total entre cursos: 80
ADVERTENCIA: el curso i1 del bloque 1 tiene tamaño 60, pero no cabe en ninguna sala.
ADVERTENCIA: el curso k1 del bloque 2 tiene tamaño 60, pero no cabe en ninguna sala.
Validación completada correctamente.

Resumen:
  Cursos bloque 1: 2
  Cursos bloque 2: 2
  Salas: 2
  Total estudiantes bloque 1: 90
  Total estudiantes bloque 2: 90
  Total estudiantes libres: 0
  Flujo total entre cursos: 90
Validación completada correctamente.

Resumen:
  Cursos bloque 1: 2
  Cursos bloque 2: 2
  Salas: 2
  Total estudiantes bloque 1: 60
  Total estudian

,test,status_esperado,status_obtenido,objetivo_esperado,objetivo_obtenido,pasa_test
0,M2_T01_factible_costo_0,OPTIMAL,OPTIMAL,0.0,0.0,True
1,M2_T02_capacidad_cambia_optimo,OPTIMAL,OPTIMAL,60.0,60.0,True
2,M2_T03_infactible_capacidad,INFEASIBLE,INFEASIBLE,NaN,NaN,True
3,M2_T04_multiples_optimos,OPTIMAL,OPTIMAL,30.0,30.0,True
4,M2_T05_estudiantes_libres,OPTIMAL,OPTIMAL,0.0,0.0,True
5,M2_T06_5_cursos_capacidades_heterogeneas,OPTIMAL,OPTIMAL,20.0,20.0,True
6,M2_T07_6_cursos_con_libres,OPTIMAL,OPTIMAL,0.0,0.0,True
7,M2_T08_8_cursos_capacidad_cambia_solucion,OPTIMAL,OPTIMAL,80.0,80.0,True
8,M2_T09_8_cursos_multiples_optimos_parciales,OPTIMAL,OPTIMAL,124.0,124.0,True


## Interpretación de los resultados

La tabla anterior resume la ejecución de los tests del Modelo 2. Para cada instancia se comparó el status esperado con el status obtenido por Gurobi, y en los casos óptimos también se comparó el valor objetivo esperado con el valor objetivo obtenido.

Los 9 tests pasaron correctamente, ya que en todos los casos la columna `pasa_test` toma el valor `True`.

Esto indica que la implementación del Modelo 2 se comporta de acuerdo con lo esperado en las instancias sintéticas diseñadas. En particular, se verificó que:

- el modelo resuelve correctamente casos factibles con costo conocido;
- las restricciones de capacidad afectan la asignación cuando corresponde;
- el modelo detecta correctamente una instancia infactible por capacidad;
- los estudiantes libres no generan costo de movilidad;
- el modelo puede manejar casos con múltiples soluciones óptimas;
- las instancias medianas también son resueltas correctamente.

El caso `M2_T03_infactible_capacidad` aparece con valor objetivo `NaN`, lo cual es correcto, ya que al ser una instancia infactible no existe una solución óptima ni un valor objetivo asociado.

## Caso demostrativo difícil: 16 cursos con flujos divididos

Después de validar el Modelo 2 con tests pequeños y medianos, se resuelve una instancia más difícil para mostrar las capacidades del modelo.

Este caso considera 16 cursos en el bloque 1, 16 cursos en el bloque 2 y 16 salas con capacidades heterogéneas. Además, los flujos de estudiantes están divididos: cada par de cursos del bloque 1 se reparte entre dos cursos del bloque 2.

La instancia está construida para que las capacidades fuercen la asignación de salas. Por ejemplo, el curso más grande solo puede ir a la sala más grande, el segundo curso más grande queda forzado a la segunda sala más grande, y así sucesivamente.

En el bloque 2, los tamaños de los cursos están intercambiados por pares, lo que obliga a una asignación distinta entre bloques. Esto genera movilidad, pero de forma controlada y explicable.

El flujo total de estudiantes es 945. De ellos, 595 permanecen en la misma sala y 350 deben cambiar de sala. Como el costo del Modelo 2 es binario, el costo esperado coincide con la cantidad de estudiantes que se mueven:


$$945 - 595 = 350$$

Por lo tanto, el resultado esperado es:

```text
Status esperado: OPTIMAL
Costo esperado: 350

In [28]:
from generar_data_modelo_2 import generar_caso_demo_modelo_2

generar_caso_demo_modelo_2()

Test generado: M2_T10_demo_16_cursos_split_capacidades


In [29]:
from modelo_2_gurobi import mostrar_caso_demo_modelo_2

In [30]:
asignacion_esperada_b1_demo = {
    "i1": "r1",
    "i2": "r2",
    "i3": "r3",
    "i4": "r4",
    "i5": "r5",
    "i6": "r6",
    "i7": "r7",
    "i8": "r8",
    "i9": "r9",
    "i10": "r10",
    "i11": "r11",
    "i12": "r12",
    "i13": "r13",
    "i14": "r14",
    "i15": "r15",
    "i16": "r16"
}

asignacion_esperada_b2_demo = {
    "k1": "r2",
    "k2": "r1",
    "k3": "r4",
    "k4": "r3",
    "k5": "r6",
    "k6": "r5",
    "k7": "r8",
    "k8": "r7",
    "k9": "r10",
    "k10": "r9",
    "k11": "r12",
    "k12": "r11",
    "k13": "r14",
    "k14": "r13",
    "k15": "r16",
    "k16": "r15"
}

resultado_demo = mostrar_caso_demo_modelo_2(
    nombre_test="M2_T10_demo_16_cursos_split_capacidades",
    objetivo_esperado=350,
    asignacion_esperada_b1=asignacion_esperada_b1_demo,
    asignacion_esperada_b2=asignacion_esperada_b2_demo
)

Validación completada correctamente.

Resumen:
  Cursos bloque 1: 16
  Cursos bloque 2: 16
  Salas: 16
  Total estudiantes bloque 1: 945
  Total estudiantes bloque 2: 945
  Total estudiantes libres: 0
  Flujo total entre cursos: 945
Test: M2_T10_demo_16_cursos_split_capacidades
Status obtenido: OPTIMAL
Objetivo esperado: 350
Objetivo obtenido: 350.0

Status correcto: True
Objetivo correcto: True
Asignación bloque 1 correcta: True
Asignación bloque 2 correcta: True

Asignación bloque 1:


,curso_b1,cantidad_estudiantes_curso_b1,sala_asignada,capacidad_sala_asignada,cumple_capacidad
0,i1,120,r1,120,True
1,i2,110,r2,110,True
2,i3,100,r3,100,True
3,i4,90,r4,90,True
4,i5,80,r5,80,True
5,i6,70,r6,70,True
6,i7,60,r7,60,True
7,i8,55,r8,55,True
8,i9,50,r9,50,True
9,i10,45,r10,45,True


Asignación bloque 2:


,curso_b2,cantidad_estudiantes_curso_b2,sala_asignada,capacidad_sala_asignada,cumple_capacidad
0,k1,110,r2,110,True
1,k2,120,r1,120,True
2,k3,90,r4,90,True
3,k4,100,r3,100,True
4,k5,70,r6,70,True
5,k6,80,r5,80,True
6,k7,55,r8,55,True
7,k8,60,r7,60,True
8,k9,45,r10,45,True
9,k10,50,r9,50,True


Todas las capacidades se respetan: True

Resumen de movilidad:


,flujo_total,estudiantes_no_se_mueven,estudiantes_se_mueven,porcentaje_no_se_mueve,porcentaje_se_mueve
0,945,595,350,0.62963,0.37037


### Interpretación del caso demostrativo

El caso demostrativo permite observar el comportamiento del Modelo 2 en una instancia más exigente que los tests anteriores. A diferencia de los tests pequeños y medianos, esta instancia contiene más cursos, más salas, tamaños heterogéneos y flujos divididos entre cursos del segundo bloque. Por lo tanto, se acerca más al tipo de situación que se quiere modelar: estudiantes que se redistribuyen entre bloques y una asignación de salas que debe considerar simultáneamente movilidad y capacidad.

El modelo entregó una solución con status `OPTIMAL`, lo que significa que Gurobi encontró una asignación factible y óptima para ambos bloques. Además, el valor objetivo obtenido coincide con el costo esperado. Esto indica que la función objetivo está calculando correctamente la movilidad total inducida por la asignación.

Las tablas de asignación muestran que todos los cursos fueron ubicados en salas con capacidad suficiente. Esto es importante porque en el Modelo 2 la principal extensión respecto al Modelo 1 es precisamente la incorporación de capacidades heterogéneas. En otras palabras, el modelo no solo busca reducir movimientos, sino que también respeta las restricciones físicas de las salas.

El resumen de movilidad permite interpretar directamente el valor objetivo. Como en el Modelo 2 el costo es binario, cada estudiante que cambia de sala aporta una unidad de costo, mientras que cada estudiante que permanece en la misma sala aporta costo cero. Por eso, el valor objetivo obtenido puede leerse como la cantidad de estudiantes que deben cambiar de sala entre el bloque 1 y el bloque 2.

Este caso también muestra una ventaja práctica del modelo: no entrega solamente un número óptimo, sino una decisión concreta de asignación para cada curso en ambos bloques. Esto permite revisar qué cursos quedan en qué salas, cuántos estudiantes tiene cada curso, qué capacidad tiene la sala asignada y cuánta movilidad genera la solución.

En conclusión, el caso demostrativo funciona como una prueba de capacidad del Modelo 2. Muestra que la formulación puede resolver una instancia más grande, con flujos divididos y capacidades heterogéneas, manteniendo resultados interpretables.